In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

print("Day 2 libraries imported successfully")

Day 2 libraries imported successfully


In [2]:
df = pd.read_csv("../data/frix_features_day1.csv")

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,origin_balance_error,dest_balance_error,is_high_risk_type,sender_txn_count,receiver_txn_count,sender_emptied_account,is_large_amount,risk_score_v1,rule_alert_v1
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,0.0,9839.64,0,1,1,0,0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,0.0,1864.28,0,1,1,0,0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,0.0,181.00,1,1,26,1,0,45,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,0.0,21363.00,1,1,27,1,0,45,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,0.0,11668.14,0,1,1,0,0,0,0


In [3]:
feature_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_error",
    "dest_balance_error",
    "is_high_risk_type",
    "sender_txn_count",
    "receiver_txn_count",
    "sender_emptied_account",
    "is_large_amount",
    "risk_score_v1"
]

X = df[feature_cols]
y = df["isFraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1048575, 13)
y shape: (1048575,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Fraud rate in train:", y_train.mean() * 100)
print("Fraud rate in test:", y_test.mean() * 100)


X_train: (838860, 13)
X_test: (209715, 13)
Fraud rate in train: 0.10895739455928284
Fraud rate in test: 0.10871897575280738


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

log_reg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

log_reg_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully")

Logistic Regression model trained successfully


In [6]:
y_pred_log = log_reg_model.predict(X_test)
y_prob_log = log_reg_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_log))
print("PR-AUC:", average_precision_score(y_test, y_prob_log))

Confusion Matrix:
[[194614  14873]
 [     6    222]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.93      0.96    209487
           1       0.01      0.97      0.03       228

    accuracy                           0.93    209715
   macro avg       0.51      0.95      0.50    209715
weighted avg       1.00      0.93      0.96    209715

ROC-AUC: 0.9883667989614396
PR-AUC: 0.4298587082566264


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully")

Random Forest model trained successfully


In [9]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("PR-AUC:", average_precision_score(y_test, y_prob_rf))

Confusion Matrix:
[[209420     67]
 [     5    223]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    209487
           1       0.77      0.98      0.86       228

    accuracy                           1.00    209715
   macro avg       0.88      0.99      0.93    209715
weighted avg       1.00      1.00      1.00    209715

ROC-AUC: 0.9971008543091776
PR-AUC: 0.984208005805978


In [10]:
model_results = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "precision_fraud": 0.01,
        "recall_fraud": 0.97,
        "false_positives": 14873,
        "false_negatives": 6,
        "roc_auc": roc_auc_score(y_test, y_prob_log),
        "pr_auc": average_precision_score(y_test, y_prob_log)
    },
    {
        "model": "Random Forest",
        "precision_fraud": 0.77,
        "recall_fraud": 0.98,
        "false_positives": 67,
        "false_negatives": 5,
        "roc_auc": roc_auc_score(y_test, y_prob_rf),
        "pr_auc": average_precision_score(y_test, y_prob_rf)
    }
])

model_results

,model,precision_fraud,recall_fraud,false_positives,false_negatives,roc_auc,pr_auc
0,Logistic Regression,0.01,0.97,14873,6,0.988367,0.429859
1,Random Forest,0.77,0.98,67,5,0.997101,0.984208


In [11]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance

,feature,importance
5,origin_balance_error,0.410004
12,risk_score_v1,0.113744
7,is_high_risk_type,0.113475
1,oldbalanceOrg,0.093915
10,sender_emptied_account,0.078927
2,newbalanceOrig,0.067788
0,amount,0.039477
3,oldbalanceDest,0.020313
6,dest_balance_error,0.019136
11,is_large_amount,0.016320


In [12]:
import joblib

joblib.dump(rf_model, "../models/random_forest_fraud_model_day2.joblib")

print("Random Forest model saved successfully")

Random Forest model saved successfully


In [13]:
import os

os.listdir("../models")

['random_forest_fraud_model_day2.joblib']

In [14]:
model_results.to_csv("../models/day2_model_comparison.csv", index=False)

print("Day 2 model comparison saved successfully")

Day 2 model comparison saved successfully


In [15]:
os.listdir("../models")

['day2_model_comparison.csv', 'random_forest_fraud_model_day2.joblib']